In [1]:
import pandas as pd
import numpy as np

import requests
import re
from bs4 import BeautifulSoup
import difflib
from tqdm import tqdm
import os
import time
from pathlib import Path

import glob

# PDF directory (Path object!)
PDF_DIR = Path(r"C:\Users\ysj28\Study PDFs")
OUT_DIR = "interconnections_datasets_auth"

# Build {ID: Path_to_pdf}
pdf_path_dict = {}

for pdf_path in PDF_DIR.glob("*.pdf"):
    # filename like: "1_Zou.pdf"
    stem = pdf_path.stem           # "1_Zou"
    study_id = int(stem.split("_")[0])  # 1
    pdf_path_dict[study_id] = pdf_path

pdf_path_dict = {str(k): v for k, v in pdf_path_dict.items()}

print("Found PDFs:", pdf_path_dict.keys())

Found PDFs: dict_keys(['10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '1', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '2', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '3', '40', '41', '42', '43', '44', '45', '46', '4', '5', '6', '7', '8', '9'])


In [2]:
# Create output directory
os.makedirs(OUT_DIR, exist_ok=True)
print("Writing outputs to:", OUT_DIR)

Writing outputs to: interconnections_datasets_auth


Preparation: Get paths to the pdfs

In [3]:
# Dictionary to track valid and invalid paths
path_status = {
    'valid': [],
    'invalid': []
}

# Loop through all entries in pdf_path_dict to check if files exist
print("Checking PDF file paths...")
for id_num, pdf_name in tqdm(pdf_path_dict.items()):
    full_path = os.path.join(PDF_DIR, pdf_name)
    
    if os.path.isfile(full_path):
        path_status['valid'].append((id_num, pdf_name))
    else:
        path_status['invalid'].append((id_num, pdf_name))

# Print results
print(f"\nResults:")
print(f"- Valid paths: {len(path_status['valid'])}/{len(pdf_path_dict)}")
print(f"- Invalid paths: {len(path_status['invalid'])}/{len(pdf_path_dict)}")

# Print details of invalid paths if any exist
if path_status['invalid']:
    print("\nInvalid paths:")
    for id_num, pdf_name in path_status['invalid']:
        print(f"ID {id_num}: {PDF_DIR}{pdf_name}")

Checking PDF file paths...


100%|██████████| 46/46 [00:00<00:00, 64962.28it/s]


Results:
- Valid paths: 46/46
- Invalid paths: 0/46


Get the Citation Matrix of out of the PDFs

In [4]:
# Load ID→BibTeX mapping (you need to prepare this file for authentication)
BIBTEX_MAPPING_FILE = os.path.join(OUT_DIR, 'bibtex_mapping_of_ids_authentication.xlsx')
if not os.path.exists(BIBTEX_MAPPING_FILE):
    raise FileNotFoundError(
        f"Missing {BIBTEX_MAPPING_FILE}. Create it (same format as interaction: columns = ['ID','Bibtex'])."
    )


df_id_bibtex = pd.read_excel(BIBTEX_MAPPING_FILE)
df_id_bibtex = df_id_bibtex[['ID','Bibtex']]
df_id_bibtex['ID'] = df_id_bibtex['ID'].astype(str)


In [5]:
# Function to extract references from a paper using GROBID and convert to BibTeX
def extract_references_to_bibtex(pdf_path):
    """Extract references from PDF using GROBID and convert to BibTeX format"""
    references = []
    
    try:
        # Read PDF content
        with open(pdf_path, "rb") as pdf_file:
            pdf_content = pdf_file.read()
        
        # Request GROBID to process references
        files = {"input": ("document.pdf", pdf_content, "application/pdf")}
        response = requests.post(
            "http://localhost:8070/api/processReferences", 
            files=files, 
            timeout=300
        )
        
        if response.status_code != 200:
            print(f"Error from GROBID API: {response.status_code}")
            return []
            
        # Parse XML response
        soup = BeautifulSoup(response.text, 'xml')
        
        for i, bibl in enumerate(soup.find_all('biblStruct')):
            # Extract key citation components
            ref_id = bibl.get('xml:id', f'ref_{i}')
            
            # Authors
            authors = []
            for author_tag in bibl.find_all('author'):
                person = author_tag.find('persName')
                if person:
                    surname = person.find('surname')
                    forename = person.find('forename')
                    
                    if surname:
                        author_name = surname.text
                        if forename:
                            author_name = f"{author_name}, {forename.text}"
                        authors.append(author_name)
            
            # Title
            title = ""
            title_tag = bibl.find('title', {'level': 'a'})
            if title_tag:
                title = title_tag.text.strip()
            
            # Year
            year = ""
            date_tag = bibl.find('date', {'type': 'published'})
            if date_tag and date_tag.get('when'):
                year = date_tag.get('when').split('-')[0]  # Extract year from date
            
            # Journal/Conference
            journal = ""
            journal_tag = bibl.find('title', {'level': 'j'})
            if journal_tag:
                journal = journal_tag.text.strip()
            else:
                book_tag = bibl.find('title', {'level': 'm'})
                if book_tag:
                    journal = book_tag.text.strip()
            
            # Volume, Issue, Pages
            volume = ""
            vol_tag = bibl.find('biblScope', {'unit': 'volume'})
            if vol_tag:
                volume = vol_tag.text.strip()
            
            issue = ""
            issue_tag = bibl.find('biblScope', {'unit': 'issue'})
            if issue_tag:
                issue = issue_tag.text.strip()
            
            pages = ""
            pages_from_tag = bibl.find('biblScope', {'unit': 'page', 'from': True})
            pages_to_tag = bibl.find('biblScope', {'unit': 'page', 'to': True})
            if pages_from_tag and pages_to_tag:
                pages = f"{pages_from_tag.get('from')}--{pages_to_tag.get('to')}"
            elif pages_from_tag:
                pages = pages_from_tag.get('from')
            
            # DOI
            doi = ""
            doi_tag = bibl.find('idno', {'type': 'DOI'})
            if doi_tag:
                doi = doi_tag.text.strip()
            
            # Create BibTeX entry
            bibtex_id = f"grobid_{ref_id.replace('b', '')}"
            bibtex_str = f"@article{{{bibtex_id},\n"
            
            if authors:
                bibtex_str += f"  author = {{{' and '.join(authors)}}},\n"
            if title:
                bibtex_str += f"  title = {{{title}}},\n"
            if journal:
                bibtex_str += f"  journal = {{{journal}}},\n"
            if year:
                bibtex_str += f"  year = {{{year}}},\n"
            if volume:
                bibtex_str += f"  volume = {{{volume}}},\n"
            if issue:
                bibtex_str += f"  number = {{{issue}}},\n"
            if pages:
                bibtex_str += f"  pages = {{{pages}}},\n"
            if doi:
                bibtex_str += f"  doi = {{{doi}}},\n"
                
            bibtex_str += "}"
            
            # Create structured citation data
            citation = {
                'bibtex': bibtex_str,
                'authors': authors,
                'title': title,
                'year': year,
                'journal': journal,
                'doi': doi,
                'raw_xml': str(bibl),
                'author_last_names': [a.split(',')[0].lower().strip() if ',' in a else a.split()[-1].lower().strip() 
                                      for a in authors]
            }
            
            references.append(citation)

    
            
    except Exception as e:
        print(f"Error extracting references: {str(e)}")

    return references

In [6]:
# Function to normalize BibTeX entries for comparison
def normalize_bibtex_for_comparison(bibtex_str):
    """Extract key information from BibTeX for comparison"""
    result = {
        'authors': [],
        'title': '',
        'year': '',
        'journal': '',
        'doi': '',
        'author_last_names': []
    }
    
    # Extract authors
    author_match = re.search(r'author\s*=\s*\{(.*?)\}', bibtex_str, re.DOTALL)
    if author_match:
        authors_str = author_match.group(1)
        authors = [a.strip() for a in authors_str.split(' and ')]
        result['authors'] = authors
        # Extract last names
        result['author_last_names'] = [a.split(',')[0].lower().strip() if ',' in a 
                                      else a.split()[-1].lower().strip() 
                                      for a in authors]
    
    # Extract title
    title_match = re.search(r'title\s*=\s*\{(.*?)\}', bibtex_str, re.DOTALL)
    if title_match:
        result['title'] = re.sub(r'[\{\}]', '', title_match.group(1).lower())
    
    # Extract year
    year_match = re.search(r'year\s*=\s*\{?(\d{4})\}?', bibtex_str)
    if year_match:
        result['year'] = year_match.group(1)
    
    # Extract journal or booktitle
    journal_match = re.search(r'journal\s*=\s*\{(.*?)\}', bibtex_str, re.DOTALL)
    if not journal_match:
        journal_match = re.search(r'booktitle\s*=\s*\{(.*?)\}', bibtex_str, re.DOTALL)
    if journal_match:
        result['journal'] = journal_match.group(1).lower()
    
    # Extract DOI
    doi_match = re.search(r'doi\s*=\s*\{(.*?)\}', bibtex_str, re.DOTALL)
    if doi_match:
        result['doi'] = doi_match.group(1).lower()
    
    return result

In [7]:
# Function to match citation with corpus papers
def match_citation_to_corpus(citation_data, corpus_entries, citing_paper_id, threshold=0.5):
    """Match a citation to papers in the corpus"""
    matches = []
    
    # Extract normalized data from citation
    citation_title = citation_data.get('title', '').lower()
    citation_title = re.sub(r'[^\w\s]', '', citation_title)
    citation_year = citation_data.get('year', '')
    citation_authors = citation_data.get('author_last_names', [])
    citation_doi = citation_data.get('doi', '').lower()
    
    for paper_id, paper_data in corpus_entries.items():
        # Skip self-citations (ADDED)
        if paper_id == citing_paper_id:
            continue
            
        score = 0
        max_score = 0
        match_details = {}
        
        # Extract normalized data from corpus entry
        paper_title = paper_data.get('title', '').lower()
        paper_title = re.sub(r'[^\w\s]', '', paper_title)
        paper_year = paper_data.get('year', '')
        paper_authors = paper_data.get('author_last_names', [])
        paper_doi = paper_data.get('doi', '').lower()
        
        # Match by DOI (highest confidence)
        if citation_doi and paper_doi and citation_doi == paper_doi:
            return [{'paper_id': paper_id, 'score': 1.0, 'match_type': 'doi'}]
        
        # Match by authors (up to 2 points)
        if citation_authors and paper_authors:
            max_score += 2
            matching_authors = set(citation_authors) & set(paper_authors)
            if matching_authors:
                author_score = min(2, len(matching_authors))
                score += author_score
                match_details['matching_authors'] = list(matching_authors)
        
        # Match by year (1 point)
        if citation_year and paper_year:
            max_score += 1
            if citation_year == paper_year:
                score += 1
                match_details['year_match'] = True
        
        # Match by title (5 points max)
        if citation_title and paper_title:
            max_score += 5
            
            # Calculate title similarity
            title_similarity = difflib.SequenceMatcher(None, citation_title, paper_title).ratio()
            
            # Check for title containment (special case)
            if (len(citation_title) > 10 and len(paper_title) > 10):
                if citation_title in paper_title or paper_title in citation_title:
                    title_similarity = max(title_similarity, 0.8)
            
            title_score = title_similarity * 5
            score += title_score
            match_details['title_similarity'] = title_similarity
        
        # Calculate final score
        if max_score > 0:
            final_score = score / max_score
            if final_score >= threshold:
                matches.append({
                    'paper_id': paper_id,
                    'score': final_score,
                    'details': match_details
                })
    
    # Sort matches by score
    matches.sort(key=lambda x: x['score'], reverse=True)
    return matches

In [8]:
# Function to build citation network with Excel-based confirmation
def build_citation_network_with_excel(df_id_bibtex, pdf_path_dict, PDF_DIR):
    """Build a citation network from papers and their references with Excel-based confirmation"""
    print("Preparing citation network analysis...")
    
    # Step 1: Create normalized corpus entries from BibTeX data
    print("Normalizing corpus BibTeX entries...")
    corpus_entries = {}
    
    for _, row in df_id_bibtex.iterrows():
        paper_id = row['ID']
        bibtex_str = row['Bibtex'] if isinstance(row['Bibtex'], str) else ""
        
        if bibtex_str:
            normalized_data = normalize_bibtex_for_comparison(bibtex_str)
            corpus_entries[paper_id] = normalized_data
    
    print(f"Normalized {len(corpus_entries)} papers in corpus")
    
    # Step 2: Initialize citation matrix
    paper_ids = list(corpus_entries.keys())
    citation_matrix = pd.DataFrame(0, index=paper_ids, columns=paper_ids)
    
    # Step 3: Extract all references first 
    print("First extracting all references from papers...")
    all_paper_references = {}
    
    # Create output directory for citations
    os.makedirs("extracted_citations", exist_ok=True)
    
    for paper_id in tqdm(corpus_entries.keys()):
        
        time.sleep(1)  # To avoid overwhelming the server
        
        if paper_id in pdf_path_dict:
            pdf_path = pdf_path_dict[paper_id]
            
            # Extract references
            print(f"\nExtracting references from paper {paper_id}...")
            references = extract_references_to_bibtex(str(pdf_path))  # to str
            print(f"Found {len(references)} references in paper {paper_id}")
            
            # Save references to file
            with open(f"extracted_citations/paper_{paper_id}_citations.bib", "w", encoding="utf-8") as f:
                for ref in references:
                    f.write(ref['bibtex'] + "\n\n")
            
            all_paper_references[paper_id] = references
    
    # Step 4: Match references to corpus and collect uncertain matches
    print("\nMatching references to corpus papers...")
    citation_details = {}
    uncertain_matches = []  # Store uncertain matches for Excel confirmation
    high_confidence_matches = []  # Store high confidence matches
    
    for citing_id, references in all_paper_references.items():
        citation_details[citing_id] = []
        
        for ref_idx, ref in enumerate(references):
            matches = match_citation_to_corpus(ref, corpus_entries, citing_id)
            
            if matches:
                best_match = matches[0]
                cited_id = best_match['paper_id']
                match_score = best_match['score']
                
                # High-confidence match (add directly)
                if match_score >= 0.7:
                    citation_matrix.loc[citing_id, cited_id] = 1
                    high_confidence_matches.append({
                        'citing_id': citing_id,
                        'cited_id': cited_id,
                        'score': match_score,
                        'citation_title': ref.get('title', ''),
                        'citation_authors': ', '.join(ref.get('authors', [])),
                        'citation_year': ref.get('year', ''),
                        'confidence': 'high',
                        'matching_authors': ', '.join(best_match.get('details', {}).get('matching_authors', [])),
                        'title_similarity': best_match.get('details', {}).get('title_similarity', 0),
                        'year_match': 'Yes' if best_match.get('details', {}).get('year_match', False) else 'No',
                        'confirmed': 'Yes'  # Auto-confirmed due to high confidence
                    })
                    print(f"✓ Paper {citing_id} cites paper {cited_id} (score: {match_score:.2f})")
                
                # Uncertain match (store for Excel confirmation)
                elif match_score >= 0.5:
                    citing_filename = pdf_path_dict.get(citing_id, f"Unknown-{citing_id}")
                    cited_filename = pdf_path_dict.get(cited_id, f"Unknown-{cited_id}")
                    
                    uncertain_matches.append({
                        'citing_id': citing_id,
                        'citing_filename': citing_filename,
                        'cited_id': cited_id,
                        'cited_filename': cited_filename,
                        'score': match_score,
                        'citation_title': ref.get('title', ''),
                        'citation_authors': ', '.join(ref.get('authors', [])),
                        'citation_year': ref.get('year', ''),
                        'confidence': 'medium',
                        'matching_authors': ', '.join(best_match.get('details', {}).get('matching_authors', [])),
                        'title_similarity': best_match.get('details', {}).get('title_similarity', 0),
                        'year_match': 'Yes' if best_match.get('details', {}).get('year_match', False) else 'No',
                        'confirmed': ''  # To be filled in Excel
                    })
                    print(f"? Uncertain match: Paper {citing_id} possibly cites paper {cited_id} (score: {match_score:.2f})")
    
    # Step 5: Export uncertain matches to Excel
    if uncertain_matches:
        print(f"\nExporting {len(uncertain_matches)} uncertain matches to Excel...")
        uncertain_df = pd.DataFrame(uncertain_matches)
        
        # Add instructions in first row
        instructions = pd.DataFrame([{
            'citing_id': 'INSTRUCTIONS',
            'citing_filename': 'Fill in the "confirmed" column with: Yes, No, or leave blank to skip',
            'cited_id': '',
            'cited_filename': '',
            'score': '',
            'citation_title': '',
            'citation_authors': '', 
            'citation_year': '',
            'confidence': '',
            'matching_authors': '',
            'title_similarity': '',
            'year_match': '',
            'confirmed': ''
        }])
        
        # Combine instructions with data
        export_df = pd.concat([instructions, uncertain_df], ignore_index=True)
        
        # Export to Excel
        excel_path = 'citation_confirmation.xlsx'
        export_df.to_excel(excel_path, index=False)
        print(f"Exported uncertain matches to {excel_path}")
        print("Please fill in the 'confirmed' column with 'Yes' or 'No' and save the file.")
        print("Then run the import_citation_confirmations() function to update the citation matrix.")
    
    # Also export high confidence matches for reference
    if high_confidence_matches:
        high_conf_df = pd.DataFrame(high_confidence_matches)
        high_conf_df.to_excel('high_confidence_citations.xlsx', index=False)
    
    return citation_matrix, citation_details, uncertain_matches

In [9]:
def import_citation_confirmations(citation_matrix):
    """Import citation confirmations from Excel and update the citation matrix"""
    confirmation_file = os.path.join(OUT_DIR, 'citation_confirmation_auth.xlsx')
    
    if not os.path.exists(confirmation_file):
        print(f"Error: Confirmation file {confirmation_file} not found.")
        return citation_matrix
    
    # Load confirmations, skipping the instruction row
    confirmations = pd.read_excel(confirmation_file, header=0, skiprows=[1])
        
    # Count statistics
    confirmed_count = 0
    rejected_count = 0
    skipped_count = 0
    
    # Process each confirmation
    for _, row in confirmations.iterrows():
        citing_id = row['citing_id']
        cited_id = row['cited_id']
        confirmation = str(row['confirmed']).strip().lower()
        
        if confirmation == 'yes':
            citation_matrix.loc[citing_id, cited_id] = 1
            confirmed_count += 1
        elif confirmation == 'no':
            # Ensure it's set to 0 (though it should already be)
            citation_matrix.loc[citing_id, cited_id] = 0
            rejected_count += 1
        else:
            skipped_count += 1
    
    # Print summary
    print(f"\nCitation Confirmation Summary:")
    print(f"- Confirmed: {confirmed_count}")
    print(f"- Rejected: {rejected_count}")
    print(f"- Skipped: {skipped_count}")
    
    # Save updated matrix
    citation_matrix.to_csv('interconnections_datasets_auth/citation_matrix_auth.csv')
    print("Updated citation matrix saved to 'interconnections_datasets_auth/citation_matrix_auth.csv'")
    
    return citation_matrix

In [10]:
# Execute the modified pipeline
citation_matrix, citation_details, uncertain_matches = build_citation_network_with_excel(
    df_id_bibtex, 
    pdf_path_dict, 
    PDF_DIR
)

# Save initial citation matrix (with only high-confidence matches)
citation_matrix.to_csv('interconnections_datasets_auth/citation_confirmation.csv')
print(citation_details)
print(uncertain_matches)

# Display instructions for the user
print("\n" + "="*80)
print("NEXT STEPS:")
print("1. Open the file 'citation_confirmation.xlsx'")
print("2. For each row, review the potential citation")
print("3. In the 'confirmed' column, enter:")
print("   - 'Yes' if it's a valid citation")
print("   - 'No' if it's not a valid citation")
print("   - Leave blank to skip")
print("4. Save the file and run the code below to update the citation matrix:")
print("   citation_matrix = import_citation_confirmations(citation_matrix)")
print("="*80)


Preparing citation network analysis...
Normalizing corpus BibTeX entries...
Normalized 46 papers in corpus
First extracting all references from papers...


  0%|          | 0/46 [00:00<?, ?it/s]


Extracting references from paper 1...


In [ ]:
# Note: There is an error for the Paper 15 since the PDF seems not well-readable. It cites none of the other studies anyways.

In [ ]:
# To run after manual confirmation:
citation_matrix = import_citation_confirmations(citation_matrix)

# Print summary statistics
num_citations = citation_matrix.sum().sum()
num_papers_with_citations = (citation_matrix.sum(axis=1) > 0).sum()
num_papers_cited = (citation_matrix.sum(axis=0) > 0).sum()

print(f"\nUpdated Citation Network Summary:")
print(f"Total citations between corpus papers: {num_citations}")
print(f"Papers that cite others in the corpus: {num_papers_with_citations}")
print(f"Papers that are cited by others: {num_papers_cited}")

Error: Confirmation file interconnections_datasets_auth\citation_confirmation_auth.xlsx not found.

Updated Citation Network Summary:
Total citations between corpus papers: 146
Papers that cite others in the corpus: 41
Papers that are cited by others: 31


In [ ]:
# Identify cases where a paper cites another with a higher ID (could be due to same year; want to avoid abvious logical errors)
forward_citations = []

# Iterate through the citation matrix
for citing_id in citation_matrix.index:
    for cited_id in citation_matrix.columns:
        # Check if this is a forward citation (earlier paper citing later paper)
        if citing_id < cited_id and citation_matrix.loc[citing_id, cited_id] == 1:
            forward_citations.append((citing_id, cited_id))

# Print the results
print(f"Found {len(forward_citations)} forward citations (earlier papers citing later papers):")
if forward_citations:
    for citing_id, cited_id in forward_citations:
        print(f"Paper {citing_id} cites paper {cited_id}")
else:
    print("No forward citations found.")

Found 46 forward citations (earlier papers citing later papers):
Paper 11 cites paper 5
Paper 12 cites paper 9
Paper 13 cites paper 3
Paper 14 cites paper 30
Paper 15 cites paper 2
Paper 15 cites paper 5
Paper 15 cites paper 6
Paper 15 cites paper 7
Paper 15 cites paper 8
Paper 15 cites paper 9
Paper 16 cites paper 5
Paper 16 cites paper 30
Paper 17 cites paper 4
Paper 17 cites paper 30
Paper 18 cites paper 30
Paper 19 cites paper 5
Paper 19 cites paper 8
Paper 19 cites paper 30
Paper 20 cites paper 21
Paper 22 cites paper 8
Paper 22 cites paper 9
Paper 24 cites paper 28
Paper 24 cites paper 33
Paper 26 cites paper 4
Paper 26 cites paper 8
Paper 26 cites paper 30
Paper 27 cites paper 30
Paper 28 cites paper 4
Paper 28 cites paper 5
Paper 28 cites paper 8
Paper 28 cites paper 30
Paper 29 cites paper 30
Paper 30 cites paper 5
Paper 31 cites paper 5
Paper 31 cites paper 8
Paper 33 cites paper 8
Paper 33 cites paper 9
Paper 34 cites paper 8
Paper 34 cites paper 9
Paper 35 cites paper 8
Pap

In [ ]:
# List of specific forward citations to remove
citations_to_correct = [
    # List the 7 specific pairs you want to remove as (citing_id, cited_id)
    # For example:
    
    (14, 30),
    (16, 30),
    (17, 30),
    (18, 30),
    (19, 30),
    (24, 28),
    (24, 28),
    (24, 33),
    (26, 30),
    (27, 30)
]

# Make a copy of the original citation matrix to avoid modifying the original
corrected_matrix = citation_matrix.copy()

# Modify the specified citations
for citing_id, cited_id in citations_to_correct:
    print(f"Removing citation: Paper {citing_id} → Paper {cited_id}")
    corrected_matrix.loc[citing_id, cited_id] = 0

# Save the corrected citation matrix
corrected_matrix.to_csv('interconnections_datasets_auth/citation_matrix_auth.csv')

# Verify the corrections
forward_citations_after = []
for citing_id in corrected_matrix.index:
    for cited_id in corrected_matrix.columns:
        if citing_id < cited_id and corrected_matrix.loc[citing_id, cited_id] == 1:
            forward_citations_after.append((citing_id, cited_id))

print(f"\nAfter correction: {len(forward_citations_after)} forward citations remain.")
if forward_citations_after:
    print("Remaining forward citations:")
    for citing_id, cited_id in forward_citations_after:
        print(f"Paper {citing_id} still cites paper {cited_id}")
else:
    print("All specified forward citations have been removed.")

Removing citation: Paper 14 → Paper 30
Removing citation: Paper 16 → Paper 30
Removing citation: Paper 17 → Paper 30
Removing citation: Paper 18 → Paper 30
Removing citation: Paper 19 → Paper 30
Removing citation: Paper 24 → Paper 28
Removing citation: Paper 24 → Paper 28
Removing citation: Paper 24 → Paper 33
Removing citation: Paper 26 → Paper 30
Removing citation: Paper 27 → Paper 30


TypeError: '<' not supported between instances of 'str' and 'int'